# 47 — DeepEval RAG Metrics
## Context Precision, Recall, Faithfulness — RAG Metrics in Code
⏱ ~15 min

**LLM-as-judge evaluation** is the state of the art for measuring RAG quality. Rather than counting word overlaps, DeepEval (Confident AI, 2024) sends each answer to a small LLM judge that decomposes claims, checks grounding, and returns a calibrated score with an explanation. By the end of this workshop you will know *what* each metric measures, *how* to run them on your own pipeline, and *what* score drops look like when things go wrong.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Metric Concepts** — What DeepEval measures and why LLM-as-judge beats ROUGE |
| 2 | **Metric Reference Table** — Formula, direction, and failure mode for each of the 5 metrics |
| 3 | **LLMTestCase** — How to construct evaluation inputs from RAG pipeline outputs |
| 4 | **RAG Pipeline** — Build the retriever + generator to produce test cases |
| 5 | **GOLDEN_DATASET** — Define expected outputs for grounded evaluation |
| 6 | **Run Evaluation** — `deepeval.evaluate()` over all 5 metrics |
| 7 | **Failure Injection** — Watch scores drop when hallucinated answers replace real ones |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` that has `requirements.txt` installed
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- No DeepEval Cloud account needed — all metrics run locally via the OpenAI judge

### Key References
> Chang, Y., Wang, X., et al. (2024). *A Survey on Evaluation of Large Language Models.* ACM Transactions on Intelligent Systems and Technology. https://arxiv.org/abs/2307.03109
>
> Es, S., James, J., et al. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation.* https://arxiv.org/abs/2309.15217
>
> DeepEval documentation: https://docs.confident-ai.com
>
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "deepeval",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "chromadb",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key_ok = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY present: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running evaluation cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Metric Concepts: Why LLM-as-Judge Wins


---

### The Problem with Lexical Metrics

Classic NLP evaluation measures string overlap — how many words in the generated answer also appear in the reference answer.

```
Reference:  "LangGraph builds stateful, multi-actor LLM applications."
Candidate:  "LangGraph is used to create multi-agent systems with state."

ROUGE-1 recall:  3/6 shared words  → 0.50
Humans:          Very accurate paraphrase → 0.95
```

The gap exists because ROUGE measures vocabulary overlap, not semantic meaning. A perfect paraphrase scores badly; a factually wrong sentence using the same words scores well.

---

### Comparison: DeepEval vs RAGAS vs ROUGE

| Approach | How It Works | Handles Paraphrase | Provides Reason | Cost per eval |
|----------|-------------|-------------------|-----------------|---------------|
| **ROUGE / BLEU** | Count word n-gram overlap | No | No | Free |
| **BERTScore** | Cosine similarity of embeddings | Partially | No | Free (local model) |
| **RAGAS** | LLM judge (faithfulness, recall, etc.) | Yes | Partial | ~$0.002/case |
| **DeepEval** | LLM judge with claim decomposition | Yes | Yes (full explanation) | ~$0.003/case |

DeepEval's edge over RAGAS: each metric decomposes the answer into discrete claims before checking them. This produces more interpretable failures — you see exactly which claim was unsupported, not just a score below threshold.

---

### How DeepEval Evaluates: The Judge Pipeline

```
  Input: question + retrieved_context + actual_output + expected_output
       |
       v
  +------------------------------------------+
  |  CLAIM DECOMPOSER (LLM call #1)          |
  |  "The capital is Paris. It has 2M        |
  |   residents. Founded in 987 AD."         |
  |   -> ["capital is Paris",                |
  |       "2M residents",                    |
  |       "founded 987 AD"]                  |
  +-------------------+----------------------+
                      |  individual claims
                      v
  +------------------------------------------+
  |  CLAIM VERIFIER (LLM call #2)            |
  |  For each claim: is it supported by      |
  |  the retrieved context?                  |
  |   -> [SUPPORTED, SUPPORTED, NOT_FOUND]   |
  +-------------------+----------------------+
                      |
                      v
  +------------------------------------------+
  |  SCORE AGGREGATOR                        |
  |  supported_claims / total_claims         |
  |  -> Faithfulness = 2/3 = 0.667          |
  |  reason: "Founding date not in context" |
  +------------------------------------------+
```

> Note: the exact number of LLM calls varies by metric. Faithfulness and ContextualRecall use the two-step decompose-then-verify pattern above. AnswerRelevancy and ContextualRelevancy use a single classification step per chunk or sentence.

---

## Part 2 — The 5 RAG Metrics: Reference Table


---

### Metric Definitions

| Metric | What It Measures | Formula | Direction | Diagnoses |
|--------|-----------------|---------|-----------|----------|
| **Faithfulness** | Are all answer claims grounded in retrieved context? | `supported_claims / total_claims` | Higher = better | Generator hallucinations |
| **AnswerRelevancy** | Does the answer actually address the question? | `relevant_sentences / total_sentences` | Higher = better | Off-topic or padded responses |
| **ContextualPrecision** | Are the most relevant chunks ranked earliest? | `sum(precision@k * relevance_k) / relevant_count` | Higher = better | Poor retrieval ranking |
| **ContextualRecall** | Is the expected answer supported by retrieved chunks? | `covered_nodes / total_expected_nodes` | Higher = better | Incomplete retrieval (low k) |
| **ContextualRelevancy** | What fraction of retrieved chunks are useful? | `relevant_chunks / total_chunks` | Higher = better | Noisy retrieval (high k) |

### Generation vs Retrieval Split

```
  RAG Pipeline
  +----------------------------------------------------------+
  |  RETRIEVER              |  GENERATOR                    |
  |                         |                               |
  |  ContextualPrecision    |  Faithfulness                 |
  |  ContextualRecall       |  AnswerRelevancy              |
  |  ContextualRelevancy    |                               |
  +----------------------------------------------------------+
```

A low **Faithfulness** score means the generator is adding information not in the context — fix the prompt or increase the grounding instruction.

A low **ContextualRecall** score means the retriever missed important chunks — increase `k` or improve chunking.

A low **ContextualPrecision** score means the retriever returns relevant content, but buries it at rank 4-5 instead of 1-2 — the LLM may overlook it.

---

### When to Use Each Metric

| Use Case | Recommended Metrics | Why |
|----------|--------------------|----- |
| Detecting hallucinations in production | Faithfulness | Direct measure of ungrounded claims |
| Tuning retrieval `k` | ContextualRecall + ContextualRelevancy | Recall rises with k; Relevancy falls with noise |
| Evaluating chunk ranking / re-ranker | ContextualPrecision | Sensitive to whether the best chunk is #1 or #5 |
| Checking answer focus | AnswerRelevancy | Catches verbose, padded, or off-topic responses |
| Full pipeline audit (CI gate) | All 5 | Single score can miss failures elsewhere |

In [ ]:
# ── 2-A: Metric summary printout ──────────────────────────────────────────────

METRIC_SUMMARY = [
    ("Faithfulness", "supported_claims / total_claims", "Generation"),
    ("AnswerRelevancy", "relevant_sentences / total_sentences", "Generation"),
    ("ContextualPrecision", "weighted precision@k over relevant chunks", "Retrieval"),
    ("ContextualRecall", "expected_output_nodes covered / total nodes", "Retrieval"),
    ("ContextualRelevancy", "relevant_chunks / total_retrieved_chunks", "Retrieval"),
]

print(f"{'Metric':<24} {'Formula':<46} {'Layer'}")
print("-" * 80)
for name, formula, layer in METRIC_SUMMARY:
    print(f"{name:<24} {formula:<46} {layer}")

print()
print("Generation metrics --> fix your prompt or model temperature")
print("Retrieval metrics  --> fix your chunking, embedding, or k")

---

## Part 3 — LLMTestCase: Structuring Inputs for Evaluation


---

DeepEval evaluates one `LLMTestCase` at a time. Each test case bundles together everything the LLM judge needs:

```python
LLMTestCase(
    input="What is LangGraph?",                     # the user's question
    actual_output="LangGraph is a library...",       # what the RAG pipeline produced
    expected_output="LangGraph is a library...",     # human-written gold standard
    retrieval_context=["chunk1", "chunk2", ...],     # what the retriever returned
)
```

### Which fields each metric uses

| Metric | input | actual_output | expected_output | retrieval_context |
|--------|-------|---------------|-----------------|-------------------|
| Faithfulness | — | required | — | required |
| AnswerRelevancy | required | required | — | — |
| ContextualPrecision | required | — | required | required |
| ContextualRecall | — | — | required | required |
| ContextualRelevancy | required | — | — | required |

You always need all four fields to run all 5 metrics in a single `evaluate()` call — otherwise you will get a missing-field error for the metrics that require it.

---

## Part 4 — Build the RAG Pipeline


---

We need a real RAG pipeline to produce `actual_output` and `retrieval_context` for each test case. This section builds the knowledge base, embeds it into ChromaDB, and runs a retriever + generator over the golden dataset.

In [ ]:
# ── 4-A: Knowledge base and golden dataset ────────────────────────────────────

from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

KNOWLEDGE_BASE = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs using graph-based workflows.",
    "LangChain provides tools for building LLM applications, including chains, agents, and retrieval systems.",
    "RAG (Retrieval-Augmented Generation) combines vector search with LLM generation to reduce hallucinations.",
    "ChromaDB is an open-source vector database for storing and retrieving embeddings at local scale.",
    "Embeddings are dense numerical representations of text that capture semantic meaning.",
    "FAISS is Facebook's library for efficient similarity search over large vector datasets.",
    "Faithfulness measures whether an LLM answer is grounded in the retrieved context (no hallucinations).",
    "Answer Relevancy measures whether the answer actually addresses the user's question.",
    "Contextual Precision measures whether the retrieved chunks that appear earlier are more relevant than later ones.",
    "Contextual Recall measures how much of the expected answer is covered by the retrieved context.",
]

# Golden dataset: (question, expected_output, pre-seeded context)
# expected_output is the human-written gold answer used by ContextualPrecision + ContextualRecall.
GOLDEN_DATASET = [
    {
        "input": "What is LangGraph?",
        "expected_output": "LangGraph is a library for building stateful, multi-actor LLM applications using graph-based workflows.",
        "context": [
            "LangGraph is a library for building stateful, multi-actor applications with LLMs using graph-based workflows."
        ],
    },
    {
        "input": "What does Faithfulness measure?",
        "expected_output": "Faithfulness measures whether the LLM answer is grounded in the retrieved context, preventing hallucinations.",
        "context": [
            "Faithfulness measures whether an LLM answer is grounded in the retrieved context (no hallucinations)."
        ],
    },
    {
        "input": "What is ChromaDB used for?",
        "expected_output": "ChromaDB is an open-source vector database for storing and querying embeddings locally.",
        "context": [
            "ChromaDB is an open-source vector database for storing and retrieving embeddings at local scale."
        ],
    },
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print(f"Golden dataset: {len(GOLDEN_DATASET)} test cases")

In [ ]:
# ── 4-B: Build the vector store and run the RAG pipeline ──────────────────────

ANSWER_PROMPT = """Answer the question using ONLY the provided context. If the context doesn't contain the answer, say 'I don't know.'

Context:
{context}

Question: {query}

Answer:"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(KNOWLEDGE_BASE, embeddings, collection_name="deepeval_nb")

rag_results = []
for case in GOLDEN_DATASET:
    # Retrieve top-3 chunks via similarity search
    docs = vectorstore.similarity_search(case["input"], k=3)
    context = [d.page_content for d in docs]
    context_text = "\n".join(f"- {c}" for c in context)

    # Generate answer using only retrieved context
    prompt = ANSWER_PROMPT.format(context=context_text, query=case["input"])
    answer = llm.invoke([HumanMessage(content=prompt)]).content

    rag_results.append({"query": case["input"], "answer": answer, "context": context})
    print(f"Q: {case['input']}")
    print(f"A: {answer[:200]}")
    print(f"Retrieved {len(context)} chunks\n")

print(f"Collected {len(rag_results)} results for evaluation.")

---

## Part 5 — GOLDEN_DATASET and LLMTestCase Assembly


---

A **golden dataset** is a curated set of (question, expected_answer) pairs that represent the ground truth for your domain. Building one manually is the highest-effort step — but it unlocks ContextualPrecision and ContextualRecall, the two retrieval metrics that require a reference answer.

### How many examples do you need?

| Use Case | Recommended Size |
|----------|------------------|
| Quick sanity check | 5–10 |
| Development iteration | 20–50 |
| CI gate (merge check) | 50–100 |
| Production regression suite | 100–500 |

Tip: Example 52 (DeepEval Synthesizer) shows how to auto-generate golden datasets from your knowledge base — it bootstraps this manual step.

In [ ]:
# ── 5-A: Assemble LLMTestCase objects from the RAG pipeline outputs ────────────

from deepeval.test_case import LLMTestCase

# Zip the golden dataset (expected outputs) with the pipeline outputs (actual outputs + retrieved context)
test_cases = [
    LLMTestCase(
        input=case["input"],
        actual_output=result["answer"],
        expected_output=case["expected_output"],
        retrieval_context=result["context"],
    )
    for case, result in zip(GOLDEN_DATASET, rag_results)
]

print(f"Assembled {len(test_cases)} LLMTestCase objects")
print()

# Inspect the first test case
tc = test_cases[0]
print("Test case 0:")
print(f"  input:            {tc.input}")
print(f"  actual_output:    {tc.actual_output[:100]}...")
print(f"  expected_output:  {tc.expected_output}")
print(f"  retrieval_context ({len(tc.retrieval_context)} chunks):")
for i, chunk in enumerate(tc.retrieval_context):
    print(f"    [{i}] {chunk[:80]}...")

---

## Part 6 — Run the Full Evaluation


---

DeepEval's `evaluate()` function runs all metrics against all test cases in a single call. Each metric uses `gpt-4o-mini` as the LLM judge (configurable via the `model=` parameter). Setting `include_reason=True` gives you a human-readable explanation of each score — critical for debugging.

**Cost estimate:** 3 test cases x 5 metrics x ~2 LLM calls each = ~30 gpt-4o-mini calls (~$0.01).

In [ ]:
# ── 6-A: Define metrics and run evaluation ────────────────────────────────────

from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig
from deepeval.models import GPTModel
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)

# threshold=0.7 means the test case PASSES if score >= 0.7, FAILS otherwise.
# include_reason=True asks the judge to explain each score.
# Explicitly allow a slower live judge response while the runner still caps the
# entire workbook at five minutes.
judge_model = GPTModel(model="gpt-4o-mini", timeout=90)
# Avoid issuing all judge requests at once; a small async pool is both rate-stable
# and fast enough for this smoke-sized dataset.
evaluation_async = AsyncConfig(run_async=True, max_concurrent=4)
metrics = [
    FaithfulnessMetric(threshold=0.7, model=judge_model, include_reason=True),
    AnswerRelevancyMetric(threshold=0.7, model=judge_model, include_reason=True),
    ContextualPrecisionMetric(threshold=0.7, model=judge_model, include_reason=True),
    ContextualRecallMetric(threshold=0.7, model=judge_model, include_reason=True),
    ContextualRelevancyMetric(threshold=0.7, model=judge_model, include_reason=True),
]

print("Running evaluation on grounded answers...")
print("(This will make ~30 gpt-4o-mini calls — expect ~30s)\n")

results = evaluate(test_cases, metrics, async_config=evaluation_async)
print("\nEvaluation complete. Review the scores and reasons above.")

In [ ]:
# ── 6-B: Print per-metric scores for each test case ───────────────────────────
# DeepEval prints a summary automatically, but this loop structures the output
# for further analysis or logging.

print("Per-test-case score breakdown:\n")
for i, tc in enumerate(test_cases):
    print(f"Test case {i}: '{tc.input}'")
    for metric in tc.metrics_metadata:
        status = "PASS" if metric.success else "FAIL"
        print(f"  [{status}] {metric.metric_name:<30} score={metric.score:.3f}  threshold={metric.threshold}")
        if metric.reason:
            reason_preview = metric.reason[:120].replace("\n", " ")
            print(f"         reason: {reason_preview}...")
    print()

---

## Part 7 — Failure Injection: Watch Scores Drop


---

The best way to understand a metric is to break it intentionally. We replace the real pipeline answers with three deliberately hallucinated answers and re-run the same evaluation.

**Expected outcome:**
- **Faithfulness** should drop sharply — the hallucinated answers contradict the retrieved context
- **AnswerRelevancy** should drop — some hallucinated answers are off-topic
- **ContextualPrecision / Recall / Relevancy** may stay similar — these measure the *retriever*, which has not changed

In [ ]:
# ── 7-A: Inject hallucinated answers ──────────────────────────────────────────
# These answers are factually wrong relative to the retrieved context.
# The retriever context is unchanged — only actual_output changes.

HALLUCINATED_ANSWERS = [
    "LangGraph is a cloud service by OpenAI that costs $99/month.",     # wrong: it's open-source
    "Faithfulness measures the length of the answer in words.",          # wrong: length != faithfulness
    "ChromaDB requires a PostgreSQL database to function.",              # wrong: it's self-contained
]

hallucinated_test_cases = [
    LLMTestCase(
        input=case["input"],
        actual_output=hallucinated,
        expected_output=case["expected_output"],
        retrieval_context=result["context"],  # same retrieval context as before
    )
    for case, result, hallucinated in zip(GOLDEN_DATASET, rag_results, HALLUCINATED_ANSWERS)
]

print("Hallucinated answers injected:")
for i, (case, answer) in enumerate(zip(GOLDEN_DATASET, HALLUCINATED_ANSWERS)):
    print(f"  [{i}] Q: {case['input']}")
    print(f"       A: {answer}")
print()

In [ ]:
# ── 7-B: Run evaluation on hallucinated answers ───────────────────────────────

print("Running evaluation on hallucinated answers...")
print("(Same metrics, same retrieved context — only actual_output has changed)\n")

hallucinated_results = evaluate(hallucinated_test_cases, metrics, async_config=evaluation_async)

print("\nEvaluation complete.")
print("\nCompare with Part 6 — Faithfulness and AnswerRelevancy should be notably lower.")
print("ContextualPrecision/Recall/Relevancy may be similar since the retriever did not change.")

In [ ]:
# ── 7-C: Side-by-side score comparison ───────────────────────────────────────


def avg_score_for_metric(test_case_list, metric_name_substr):
    """Average the score for any metric whose name contains metric_name_substr."""
    scores = []
    for tc in test_case_list:
        for m in tc.metrics_metadata:
            if metric_name_substr.lower() in m.metric_name.lower():
                scores.append(m.score)
    return sum(scores) / len(scores) if scores else None


metric_keys = [
    ("Faithfulness", "Faithfulness"),
    ("AnswerRelevancy", "AnswerRelevancy"),
    ("ContextualPrecision", "ContextualPrecision"),
    ("ContextualRecall", "ContextualRecall"),
    ("ContextualRelevancy", "ContextualRelevancy"),
]

print(f"{'Metric':<28} {'Grounded':>10} {'Hallucinated':>14} {'Drop':>8}")
print("-" * 64)

for label, key in metric_keys:
    grounded = avg_score_for_metric(test_cases, key)
    hallucinated = avg_score_for_metric(hallucinated_test_cases, key)
    if grounded is not None and hallucinated is not None:
        drop = hallucinated - grounded
        drop_str = f"{drop:+.3f}"
        print(f"{label:<28} {grounded:>10.3f} {hallucinated:>14.3f} {drop_str:>8}")
    else:
        print(f"{label:<28} {'N/A':>10} {'N/A':>14} {'N/A':>8}")

print()
print("Negative drop = hallucinated answers scored LOWER (as expected).")
print("Minimal drop on contextual metrics = retriever was not changed.")

---

## Exercises

Work through these after completing Parts 1–7. The answer key is at the bottom of this notebook.

---

**Exercise 1 — Retrieval depth**

Change `k=3` to `k=1` in cell 4-B (the `similarity_search` call). Re-run the evaluation pipeline from Part 4 onward.

1. Which metric drops the most?
2. Which metrics are unaffected?
3. Why does ContextualRecall drop but ContextualRelevancy may rise?

---

**Exercise 2 — Context poisoning**

Add a wrong fact to `KNOWLEDGE_BASE`:
```python
"LangGraph is a cloud service by OpenAI priced at $99/month."
```
Rebuild the vector store and re-run. Observe what happens to Faithfulness — the model correctly cites the poisoned context, but the citation itself is wrong. What does this tell you about the limits of Faithfulness as a safety metric?

---

**Exercise 3 — Custom golden dataset**

Write your own 5-question golden dataset drawn from `KNOWLEDGE_BASE`. Each entry should have `input`, `expected_output`, and `context`. Run all 5 metrics. Which metric is hardest to score above 0.9 on your dataset?

---

**Exercise 4 — Threshold sensitivity**

Change `threshold=0.7` to `threshold=0.9` for all 5 metrics in cell 6-A. Re-run on the grounded test cases. Which metric fails first (i.e., scores below 0.9 even for correct answers)? What does this tell you about the calibration of that metric?

---

## What's Next?

You can now score any RAG pipeline on 5 LLM-judge metrics and diagnose whether failures come from the retriever or the generator. Here is where to go deeper:

### Stay in the DeepEval series
- **Example 48 — DeepEval Hallucination and Bias** (`../48-deepeval-hallucination-bias/`) — adds safety-oriented metrics: Hallucination, Bias, Toxicity. Use these when your RAG system generates user-facing content that needs safety guardrails, not just factual accuracy.
- **Example 49 — DeepEval G-Eval Custom Metrics** (`../49-deepeval-geval-custom/`) — build your own metric with a natural language rubric. Essential when none of the 5 built-in metrics captures what matters for your use case (e.g., "is the tone formal?").
- **Example 52 — DeepEval Synthesizer** (`../52-deepeval-synthesizer/`) — auto-generate golden datasets from your knowledge base. Removes the manual annotation bottleneck from Exercise 3.

### Compare evaluation frameworks
- **Example 16 — RAG Eval with RAGAS** (`../16-rag-eval-notebook/`) — run the same pipeline through RAGAS and compare scores. RAGAS uses a slightly different decomposition approach; comparing both frameworks on the same data reveals where they agree and disagree.
- **Example 29 — LLM Judge Harness** (`../29-llm-judge-harness/`) — build a custom judge from scratch without DeepEval. Useful when you need full control over the judge prompt, scoring rubric, or cost.

### Further reading
- DeepEval metric internals: https://docs.confident-ai.com/docs/metrics-introduction
- Chang et al. (2024). *A Survey on Evaluation of LLMs.* https://arxiv.org/abs/2307.03109
- Es et al. (2023). *RAGAS.* https://arxiv.org/abs/2309.15217

---

*Workshop complete. The next natural step is Example 48 (safety metrics) or Example 52 (auto-generating the golden dataset you just wrote by hand).*

---

## Exercise Answer Key

Attempt each exercise yourself before reading the solutions.

---

### Exercise 1 Answer — Retrieval Depth (k=1)

**Expected observations:**

1. **ContextualRecall drops most sharply.** With only 1 retrieved chunk, the context may not cover all nodes in the expected output. If the expected answer mentions two facts and only one chunk is returned, recall is at most 0.5.

2. **ContextualRelevancy may rise.** With 1 chunk instead of 3, the single retrieved chunk is typically the most relevant one. The ratio `relevant_chunks / total_chunks` goes from (say) 2/3 to 1/1.

3. **Faithfulness is largely unaffected** (assuming the single chunk still supports the answer). The generator can only hallucinate beyond what is provided — if the retrieved chunk was sufficient, faithfulness stays high.

**Key insight:** Recall and Relevancy are in tension — increasing k improves recall but risks diluting relevancy with noisy chunks. The optimal k depends on your knowledge base density.

In [ ]:
# Exercise 1 — predicted score direction as k changes

print("Predicted score direction as k increases:")
print(f"  {'Metric':<28} {'k=1':>8} {'k=3':>8} {'k=5':>8}")
print("  " + "-" * 56)
predictions = [
    ("Faithfulness", "stable", "stable", "stable"),
    ("AnswerRelevancy", "stable", "stable", "stable"),
    ("ContextualPrecision", "high", "medium", "lower"),
    ("ContextualRecall", "low", "medium", "higher"),
    ("ContextualRelevancy", "high", "medium", "lower"),
]
for row in predictions:
    print(f"  {row[0]:<28} {row[1]:>8} {row[2]:>8} {row[3]:>8}")
print()
print("Run the evaluation at each k value to verify these predictions.")

### Exercise 2 Answer — Context Poisoning

**Expected observation:** Faithfulness stays HIGH even though the answer is factually wrong.

**Why this matters:** Faithfulness only checks whether the answer is *grounded in the retrieved context* — it does not check whether the retrieved context itself is correct. If you poison the knowledge base, the model faithfully repeats the poison and scores 1.0.

**What this tells you about the limits of Faithfulness:** It is a necessary but not sufficient safety metric. You can have high Faithfulness and still produce harmful misinformation if the source documents are wrong or manipulated. Faithfulness prevents the model from *adding* hallucinations beyond the context, but it cannot catch *inherited* misinformation from the knowledge base itself.

**Production implication:** Document provenance and quality control at ingestion time are as important as your evaluation metrics. Consider a separate factual accuracy check against a trusted reference source for high-stakes domains.

### Exercise 3 Answer — Custom Golden Dataset

**Sample 5-question golden dataset based on `KNOWLEDGE_BASE`:**

In [ ]:
# Exercise 3 — sample answer

CUSTOM_GOLDEN_DATASET = [
    {
        "input": "What is RAG?",
        "expected_output": "RAG combines vector search with LLM generation to reduce hallucinations.",
        "context": [
            "RAG (Retrieval-Augmented Generation) combines vector search with LLM generation to reduce hallucinations."
        ],
    },
    {
        "input": "What are embeddings?",
        "expected_output": "Embeddings are dense numerical representations of text that capture semantic meaning.",
        "context": ["Embeddings are dense numerical representations of text that capture semantic meaning."],
    },
    {
        "input": "What does Answer Relevancy measure?",
        "expected_output": "Answer Relevancy measures whether the answer actually addresses the user's question.",
        "context": ["Answer Relevancy measures whether the answer actually addresses the user's question."],
    },
    {
        "input": "What is FAISS used for?",
        "expected_output": "FAISS is used for efficient similarity search over large vector datasets.",
        "context": [
            "FAISS is Facebook's library for efficient similarity search over large vector datasets."
        ],
    },
    {
        "input": "What does LangChain provide?",
        "expected_output": "LangChain provides tools for building LLM applications including chains, agents, and retrieval systems.",
        "context": [
            "LangChain provides tools for building LLM applications, including chains, agents, and retrieval systems."
        ],
    },
]

# Run the RAG pipeline over this custom dataset
custom_rag_results = []
for case in CUSTOM_GOLDEN_DATASET:
    docs = vectorstore.similarity_search(case["input"], k=3)
    context = [d.page_content for d in docs]
    context_text = "\n".join(f"- {c}" for c in context)
    prompt = ANSWER_PROMPT.format(context=context_text, query=case["input"])
    answer = llm.invoke([HumanMessage(content=prompt)]).content
    custom_rag_results.append({"query": case["input"], "answer": answer, "context": context})

custom_test_cases = [
    LLMTestCase(
        input=case["input"],
        actual_output=result["answer"],
        expected_output=case["expected_output"],
        retrieval_context=result["context"],
    )
    for case, result in zip(CUSTOM_GOLDEN_DATASET, custom_rag_results)
]

print(f"Custom golden dataset: {len(custom_test_cases)} test cases ready.")
print("Uncomment the next line to run the full evaluation (~$0.02):")
# custom_results = evaluate(custom_test_cases, metrics)

### Exercise 4 Answer — Threshold Sensitivity

**Expected observation:** With `threshold=0.9`, **ContextualPrecision** typically fails first on well-grounded answers.

**Why ContextualPrecision is hardest to push above 0.9:** Precision@k is sensitive to ranking — if the most relevant chunk is ranked 2nd instead of 1st (due to small embedding distance differences), the score drops meaningfully. For a 3-chunk retrieval window, having the best chunk at position 2 can drop precision from ~1.0 to ~0.75.

**Production implication:** If you add a re-ranker (cross-encoder) after the initial similarity search, ContextualPrecision typically improves the most — the re-ranker pulls the most relevant chunk to rank 1. See Example 54 (Reranking RAG) for a working implementation.

**Calibration note:** A threshold of 0.9 is appropriate for high-stakes production gates (medical, legal). For rapid iteration during development, 0.7 is more practical — 0.9 will generate many false failures on genuinely good answers due to natural LLM judge variance (~0.05-0.10 on repeated runs).